In [1]:
"""
Week 7 - Function 1
Bayesian NN surrogate with Bayesian linear head (Pyro VI)
+ hyperparameter tuning via Random Search + 4-fold CV
using the supplied 2D inputs and 1D outputs (16 datapoints).

Requirements (local):
    pip install numpy torch pyro-ppl scikit-learn
"""

import math
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


# ------------------------ 1. Base Config ------------------------

RANDOM_SEED = 123
INPUT_DIM = 2
HIDDEN_SIZES = [64, 64]       # keep fixed for Week 7 to focus tuning where it matters most

N_EPOCHS_FEATURE = 2000
LR_FEATURE = 1e-3

# ---- Tuning search sizes ----
N_TUNING_TRIALS = 20          # random search trials
N_FOLDS = 4                   # 4-fold CV

# ---- Candidate search ----
N_CANDIDATES = 30000          # increased for better global coverage
TOP_K = 5

# default fallbacks (used only if tuning fails)
DEFAULT_PRIOR_SCALE = 1.0
DEFAULT_LR_VI = 5e-3
DEFAULT_N_STEPS_VI = 3000
DEFAULT_N_PRED_SAMPLES = 256
DEFAULT_XI = 0.01


# ------------------------ 2. Data ------------------------

def load_data():
    X_raw = np.array([
        [0.31940389, 0.76295937],
        [0.57432921, 0.87989810],
        [0.73102363, 0.73299988],
        [0.84035342, 0.26473161],
        [0.65011406, 0.68152635],
        [0.41043714, 0.14755430],
        [0.31269116, 0.07872278],
        [0.68341817, 0.86105746],
        [0.08250725, 0.40348751],
        [0.88388983, 0.58225397],
        [0.88389,    0.98389],
        [0.37454,    0.950713],
        [0.382224,   0.951319],
        [0.782778,   0.793329],
        [0.030500,   0.037300],
        [0.646168,   0.172681],
    ], dtype=np.float64)

    y_raw = np.array([
        1.32267704e-79,
        1.03307824e-46,
        7.71087511e-16,
        3.34177101e-124,
        -3.60606264e-03,
        -2.15924904e-54,
        -2.08909327e-91,
        2.53500115e-40,
        3.60677119e-81,
        6.22985647e-48,
        9.59033053e-135,
        -1.56227724e-117,
        -4.77166224e-115,
        7.535209723645751e-36,
        1.6357533426693436e-209,
        6.327028545366271e-79,
    ], dtype=np.float64)

    assert X_raw.shape[1] == INPUT_DIM
    return X_raw, y_raw


# ------------------------ 3. Feature extractor (deterministic) ------------------------

class FeatureExtractor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        self.net = nn.Sequential(*layers)
        self.output_dim = prev_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DeterministicRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        self.feature_extractor = FeatureExtractor(input_dim, hidden_sizes)
        self.head = nn.Linear(self.feature_extractor.output_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.feature_extractor(x)
        return self.head(feats).squeeze(-1)


def train_feature_extractor(X: torch.Tensor, y: torch.Tensor) -> FeatureExtractor:
    torch.manual_seed(RANDOM_SEED)
    model = DeterministicRegressor(INPUT_DIM, HIDDEN_SIZES)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_FEATURE)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(N_EPOCHS_FEATURE):
        optimizer.zero_grad()
        preds = model(X)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

    feature_extractor = FeatureExtractor(INPUT_DIM, HIDDEN_SIZES)
    feature_extractor.load_state_dict(model.feature_extractor.state_dict())
    return feature_extractor


# ------------------------ 4. Bayesian linear head (Pyro model) ------------------------

@dataclass
class BayesianHeadConfig:
    prior_scale: float


def make_bayesian_model(feature_extractor: FeatureExtractor, cfg: BayesianHeadConfig):
    def model(x, y=None):
        pyro.module("feature_extractor", feature_extractor, update_module_params=False)
        feats = feature_extractor(x)
        H = feats.size(-1)

        weight = pyro.sample(
            "weight",
            dist.Normal(x.new_zeros(H), cfg.prior_scale * x.new_ones(H)).to_event(1)
        )
        bias = pyro.sample("bias", dist.Normal(x.new_tensor(0.0), cfg.prior_scale))

        # slightly tighter likelihood prior for stability on small data
        sigma = pyro.sample("sigma", dist.HalfCauchy(x.new_tensor(0.05)))

        mean = (feats * weight).sum(dim=-1) + bias

        with pyro.plate("data", x.size(0)):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

    return model


def train_bayesian_head(model, X: torch.Tensor, y: torch.Tensor, lr_vi: float, n_steps_vi: int):
    pyro.clear_param_store()
    guide = AutoDiagonalNormal(model)
    optimizer = Adam({"lr": lr_vi})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

    for _ in range(n_steps_vi):
        svi.step(X, y)

    return guide


# ------------------------ 5. Acquisition functions ------------------------

def normal_cdf(x: torch.Tensor) -> torch.Tensor:
    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))


def normal_pdf(x: torch.Tensor) -> torch.Tensor:
    return (1.0 / math.sqrt(2.0 * math.pi)) * torch.exp(-0.5 * x**2)


def compute_ei_and_pi(samples: torch.Tensor, best_y: float, xi: float):
    mean = samples.mean(dim=0)
    std = samples.std(dim=0) + 1e-9
    gamma = (mean - best_y - xi) / std
    ei = (mean - best_y - xi) * normal_cdf(gamma) + std * normal_pdf(gamma)
    ei = torch.clamp(ei, min=0.0)
    pi = normal_cdf((mean - best_y - xi) / std)
    return ei, pi, mean, std


# ------------------------ 6. Hyperparameter tuning (Random Search + CV) ------------------------

def mc_predictive_mean_std(model, guide, X: torch.Tensor, n_samples: int):
    predictive = Predictive(model, guide=guide, num_samples=n_samples, return_sites=("obs",))
    with torch.no_grad():
        pred = predictive(X)["obs"]  # (S, N)
    return pred.mean(dim=0), pred.std(dim=0) + 1e-9


def cv_score(feature_extractor: FeatureExtractor,
             X_all: torch.Tensor,
             y_all: torch.Tensor,
             prior_scale: float,
             lr_vi: float,
             n_steps_vi: int,
             n_pred_samples: int):
    """
    Score = average predictive log likelihood on held-out folds (higher is better).
    Uses Normal(μ, σ) estimated from posterior predictive MC.
    """
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    scores = []

    for train_idx, val_idx in kf.split(X_all.cpu().numpy()):
        X_tr, y_tr = X_all[train_idx], y_all[train_idx]
        X_va, y_va = X_all[val_idx], y_all[val_idx]

        cfg = BayesianHeadConfig(prior_scale=prior_scale)
        model = make_bayesian_model(feature_extractor, cfg)
        guide = train_bayesian_head(model, X_tr, y_tr, lr_vi=lr_vi, n_steps_vi=n_steps_vi)

        mu, sd = mc_predictive_mean_std(model, guide, X_va, n_samples=n_pred_samples)
        # Gaussian log likelihood per point
        ll = (-0.5 * torch.log(2 * math.pi * sd**2) - 0.5 * ((y_va - mu) ** 2) / (sd**2)).mean()
        scores.append(ll.item())

    return float(np.mean(scores))


def random_log_uniform(rng, low, high):
    # sample in log space
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))


def tune_hyperparameters(feature_extractor: FeatureExtractor,
                         X_all: torch.Tensor,
                         y_all: torch.Tensor):
    """
    Random search over:
      prior_scale ~ logU(0.1, 3.0)
      lr_vi       ~ logU(1e-3, 1e-2)
      n_steps_vi  in {1500, 3000, 5000}
      n_pred_samp in {128, 256, 384}
      xi          ~ logU(1e-4, 5e-2)   (used later in acquisition; tuned with model params)
    """
    rng = np.random.default_rng(RANDOM_SEED)

    steps_choices = [1500, 3000, 5000]
    pred_choices = [128, 256, 384]

    best = {
        "score": -1e18,
        "prior_scale": DEFAULT_PRIOR_SCALE,
        "lr_vi": DEFAULT_LR_VI,
        "n_steps_vi": DEFAULT_N_STEPS_VI,
        "n_pred_samples": DEFAULT_N_PRED_SAMPLES,
        "xi": DEFAULT_XI,
    }

    for t in range(N_TUNING_TRIALS):
        prior_scale = random_log_uniform(rng, 0.1, 3.0)
        lr_vi = random_log_uniform(rng, 1e-3, 1e-2)
        n_steps_vi = int(rng.choice(steps_choices))
        n_pred_samples = int(rng.choice(pred_choices))
        xi = random_log_uniform(rng, 1e-4, 5e-2)

        try:
            score = cv_score(
                feature_extractor, X_all, y_all,
                prior_scale=prior_scale,
                lr_vi=lr_vi,
                n_steps_vi=n_steps_vi,
                n_pred_samples=n_pred_samples
            )
        except Exception:
            # if a trial is numerically unstable, skip it
            continue

        if score > best["score"]:
            best.update({
                "score": score,
                "prior_scale": prior_scale,
                "lr_vi": lr_vi,
                "n_steps_vi": n_steps_vi,
                "n_pred_samples": n_pred_samples,
                "xi": xi,
            })

    return best


# ------------------------ 7. Propose next point ------------------------

def propose_next_point(model,
                       guide,
                       X_train_scaled: torch.Tensor,
                       y_train_scaled: torch.Tensor,
                       scaler_y: StandardScaler,
                       xi: float,
                       n_pred_samples: int):
    X_candidates = torch.rand((N_CANDIDATES, INPUT_DIM))

    predictive = Predictive(model, guide=guide, num_samples=n_pred_samples, return_sites=("obs",))
    with torch.no_grad():
        samples_scaled = predictive(X_candidates)["obs"]

    best_y_scaled = float(y_train_scaled.max().item())
    ei, pi, mean_scaled, std_scaled = compute_ei_and_pi(samples_scaled, best_y_scaled, xi=xi)

    idx_best = int(torch.argmax(ei).item())
    next_x = X_candidates[idx_best].cpu().numpy()

    mean_scaled_np = mean_scaled.cpu().numpy()
    std_scaled_np = std_scaled.cpu().numpy()

    mean_raw = scaler_y.inverse_transform(mean_scaled_np.reshape(-1, 1)).squeeze(-1)
    std_raw = std_scaled_np * scaler_y.scale_[0]

    best_y_raw = float(scaler_y.inverse_transform(np.array(best_y_scaled).reshape(-1, 1)).squeeze())

    topk_vals, topk_idx = torch.topk(ei, k=min(TOP_K, N_CANDIDATES))
    top_candidates = []
    for rank, (ei_val, j) in enumerate(zip(topk_vals, topk_idx)):
        j = int(j.item())
        x_j = X_candidates[j].cpu().numpy()
        top_candidates.append({
            "rank": rank + 1,
            "x": x_j,
            "ei_scaled": float(ei_val.item()),
            "pred_mean": float(mean_raw[j]),
            "pred_std": float(std_raw[j]),
            "prob_improvement": float(pi[j].item()),
        })

    info = {
        "next_x": next_x,
        "next_pred_mean": float(mean_raw[idx_best]),
        "next_pred_std": float(std_raw[idx_best]),
        "next_prob_improvement": float(pi[idx_best].item()),
        "best_y_raw": best_y_raw,
        "top_candidates": top_candidates,
    }
    return next_x, info


# ------------------------ 8. Main pipeline ------------------------

def main():
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    X_raw, y_raw = load_data()
    print(f"Loaded X_raw shape: {X_raw.shape}, y_raw shape: {y_raw.shape}")

    scaler_y = StandardScaler()
    y_scaled_np = scaler_y.fit_transform(y_raw.reshape(-1, 1)).astype(np.float32).ravel()

    X_t = torch.from_numpy(X_raw.astype(np.float32))
    y_t = torch.from_numpy(y_scaled_np.astype(np.float32))

    print("\n=== Pretraining deterministic feature extractor ===")
    feature_extractor = train_feature_extractor(X_t, y_t)
    feature_extractor.eval()
    print("Feature extractor trained.")

    print("\n=== Hyperparameter tuning (Random Search + 4-fold CV) ===")
    best_hp = tune_hyperparameters(feature_extractor, X_t, y_t)

    print("Best hyperparameters found:")
    print(f"  CV score (avg log-lik): {best_hp['score']:.6f}")
    print(f"  prior_scale:    {best_hp['prior_scale']:.6g}")
    print(f"  lr_vi:          {best_hp['lr_vi']:.6g}")
    print(f"  n_steps_vi:     {best_hp['n_steps_vi']}")
    print(f"  n_pred_samples: {best_hp['n_pred_samples']}")
    print(f"  xi (EI/PI):     {best_hp['xi']:.6g}")

    print("\n=== Training final Bayesian head on ALL data (best HP) ===")
    bayes_cfg = BayesianHeadConfig(prior_scale=best_hp["prior_scale"])
    bayes_model = make_bayesian_model(feature_extractor, bayes_cfg)
    guide = train_bayesian_head(
        bayes_model, X_t, y_t,
        lr_vi=best_hp["lr_vi"],
        n_steps_vi=best_hp["n_steps_vi"]
    )
    print("Final Bayesian head training complete.")

    print("\n=== Proposing next query point (EI with tuned xi) ===")
    next_x, info = propose_next_point(
        bayes_model,
        guide,
        X_t,
        y_t,
        scaler_y,
        xi=best_hp["xi"],
        n_pred_samples=best_hp["n_pred_samples"]
    )

    print("\n=== CURRENT BEST (from observed data) ===")
    print(f"Best observed y (original scale): {info['best_y_raw']:.6g}")

    print("\n=== PROPOSED NEXT QUERY POINT ===")
    print(f"x_next (in [0,1]^2): {info['next_x']}")
    print(f"Predicted y at x_next (mean, original scale): {info['next_pred_mean']:.6g}")
    print(f"Predictive std at x_next (original scale): {info['next_pred_std']:.6g}")
    print(f"Probability of improvement over current best: {info['next_prob_improvement'] * 100:.2f}%")

    print("\n=== TOP CANDIDATES (by EI, in [0,1]^2) ===")
    for cand in info["top_candidates"]:
        x = cand["x"]
        print(
            f"Rank {cand['rank']:>2d}: x={x}, "
            f"pred_mean={cand['pred_mean']:.6g}, "
            f"pred_std={cand['pred_std']:.6g}, "
            f"PI={cand['prob_improvement']*100:5.2f}%, "
            f"EI_scaled={cand['ei_scaled']:.4g}"
        )

    print("\nDone.")


if __name__ == "__main__":
    main()


Loaded X_raw shape: (16, 2), y_raw shape: (16,)

=== Pretraining deterministic feature extractor ===
Feature extractor trained.

=== Hyperparameter tuning (Random Search + 4-fold CV) ===
Best hyperparameters found:
  CV score (avg log-lik): -41.214822
  prior_scale:    1.65008
  lr_vi:          0.00163592
  n_steps_vi:     3000
  n_pred_samples: 384
  xi (EI/PI):     0.00501406

=== Training final Bayesian head on ALL data (best HP) ===
Final Bayesian head training complete.

=== Proposing next query point (EI with tuned xi) ===

=== CURRENT BEST (from observed data) ===
Best observed y (original scale): -2.30211e-12

=== PROPOSED NEXT QUERY POINT ===
x_next (in [0,1]^2): [0.54668313 0.5623147 ]
Predicted y at x_next (mean, original scale): 7.27159e-05
Predictive std at x_next (original scale): 0.00812172
Probability of improvement over current best: 50.34%

=== TOP CANDIDATES (by EI, in [0,1]^2) ===
Rank  1: x=[0.54668313 0.5623147 ], pred_mean=7.27159e-05, pred_std=0.00812172, PI=50.